In [25]:
import pandas as pd

df = pd.read_csv("scraped_data_en.csv")
df.head()

,link,title,date,text
0,https://www.primeminister.am/en/foreign-visits...,Prime Minister Nikol Pashinyan's working visit...,24.07.2025 - 25.07.2025,Nikol Pashinyan arrived in the Altai Republic ...
1,https://www.primeminister.am/en/foreign-visits...,Prime Minister Nikol Pashinyan's working visit...,14.07.2025 - 14.07.2025,Prime Minister Nikol Pashinyan has arrived in ...
2,https://www.primeminister.am/en/foreign-visits...,Prime Minister Nikol Pashinyan's working visit...,14.07.2025 - 14.07.2025,Prime Minister Nikol Pashinyan has arrived in ...
3,https://www.primeminister.am/en/foreign-visits...,Prime Minister Nikol Pashinyan's working visit...,09.07.2025 - 10.07.2025,Prime Minister Nikol Pashinyan has arrived in...
4,https://www.primeminister.am/en/foreign-visits...,Prime Minister Nikol Pashinyan's working visit...,20.06.2025 - 20.06.2025,Prime Minister Nikol Pashinyan has arrived in...


In [35]:
from openai import OpenAI
import time
import os

from dotenv import load_dotenv

In [36]:
load_dotenv(override=True)
api_key = os.environ['OPENAI_API_KEY_LEDE']

client = OpenAI(api_key=api_key)

In [37]:
from pydantic import BaseModel, Field
from openai import OpenAI
from typing_extensions import Literal
from typing import Optional

class VisitDetails(BaseModel):
    country: str = Field(description="Country visited")
    cities: list[str] = Field(description="Cities visited in the country")

prompt = """
You are an expert at geopolitics and data extraction. The following is a text about Armenian PM Nikol Pashinyan's official visit abroad.

Extract the following details:

1. The country visited
2. The cities visited in that country

Return the details in JSON format with the following structure:

{
    "country": "Country Name",
    "cities": ["City1", "City2", ...]
}

Example:
{
    "country": "France",
    "cities": ["Paris", "Lyon"]
}

## Important Notes:
- If the country is not clear or ambiguous, return "UNKNOWN" for the country and an empty list for cities.
- If multiple countries are mentioned, return the first one.
- If no cities are mentioned, return an empty list for cities.
- If the country is mentioned but the city is not, return the country and an empty list for cities.
- Use the common short English name for the country, which may not be the formal official name.
    - For example, use "South Korea" instead of "Republic of Korea".
    - For example, use "Russia" instead of "Russian Federation".
    - For example, use USA" instead of "United States".
"""

def get_visit_details(text: str):
    completion = client.chat.completions.parse(
        model="gpt-4.1-nano",
        messages=[
            {"role": "system", "content": prompt},
            {"role": "user", "content": text}
        ],
        temperature=0,
        response_format=VisitDetails
    )

    result = completion.choices[0].message.parsed
    return result


In [38]:
get_visit_details(df.iloc[0]['text'])

VisitDetails(country='Russia', cities=['Gorno-Altaysk'])

In [39]:
from tqdm.auto import tqdm
tqdm.pandas()

df = df.assign(
    **df['text']
    .progress_apply(get_visit_details)
    .apply(
        lambda result: pd.Series(result.model_dump()))
    
)

  0%|          | 0/135 [00:00<?, ?it/s]

In [40]:
df['country'].value_counts()

country
Russia            33
France            15
Belgium           12
Georgia           10
Kazakhstan         8
Germany            7
Iran               5
USA                5
Kyrgyzstan         5
Czech Republic     3
Turkey             2
Switzerland        2
Belarus            2
Tajikistan         2
Austria            1
Luxembourg         1
China              1
Vietnam            1
Singapore          1
Turkmenistan       1
Italy              1
Lithuania          1
Netherlands        1
Qatar              1
Moldova            1
Tunisia            1
Iceland            1
Albania            1
Spain              1
UAE                1
Greece             1
Egypt              1
Denmark            1
UK                 1
Hungary            1
Vatican            1
Estonia            1
Lebanon            1
Name: count, dtype: int64

In [31]:
pd.pivot_table(
    df,
    values='link',            # the column to "count"
    index='country',          # rows
    aggfunc='count',           # counts non-null values (like COUNTA)
).sort_values(by='link', ascending=False)

,link
country,
Russia,33
France,15
Belgium,12
Georgia,10
Kazakhstan,8
Germany,7
USA,5
Kyrgyzstan,5
Iran,5
